# Pipeline xây dựng bộ từ điển Lexicon (Loughran–McDonald) cho tiếng Việt

Notebook này gộp lại **3 bước xử lý tuần tự**, trước đây được tách thành 3 file `.py`
riêng biệt (`extract_master_dictionary_categories.py`, `translate_master_dictionary_seeds.py`,
`check_vi_seeds_against_corpus.py`). Gộp vào 1 notebook giúp chạy từng bước, xem kết quả
trung gian ngay bên dưới mỗi cell, và dễ kiểm soát/gỡ lỗi hơn so với 3 file rời rạc.

## Luồng xử lý tổng quan

```
Bước 1: Trích danh mục gốc (tiếng Anh)
  Loughran-McDonald Master Dictionary (CSV)
        |  lọc theo 7 cột flag: Negative, Positive, Uncertainty,
        |  Litigious, Strong_Modal, Weak_Modal, Constraining
        v
  data/categories/{category}_master_dictionary.csv   (7 file)

Bước 2: Dịch & gộp thành seed tiếng Việt
  Bản đồ thủ công: cụm từ tiếng Việt -> nhóm từ gốc tiếng Anh
        |  đối chiếu ngược với CSV ở Bước 1 để tránh gõ nhầm chính tả
        v
  vi_seeds/{category}_word.txt                 (seed dùng cho pipeline)
  data/categories/{category}_vi_mapping.csv    (bản đồ đầy đủ, để tra cứu)

Bước 3: Kiểm tra seed tiếng Việt có xuất hiện trong corpus thật không
  Corpus tin tức đã tokenize (equity_news_tokenized_underthesea.parquet)
        |  build n-gram 1-3, đối chiếu với từng seed
        v
  data/corpus_check/vi_seeds_corpus_presence.csv
```

**Vì sao cần Bước 3?** Seed được dịch thủ công ở Bước 2 có thể không khớp đúng cách mà
tokenizer tiếng Việt tách từ trong thực tế (ví dụ viết `rủi_ro` nhưng corpus lại tokenize
ra cụm khác), nên cần đối chiếu lại với corpus thật trước khi dùng seed để tính PMI / xây
dictionary sentiment đầy đủ ở bước kế tiếp.

**Cách chạy**: chạy tuần tự từng cell từ trên xuống dưới (Bước 1 -> Bước 2 -> Bước 3).
Mỗi bước ghi output ra đĩa nên có thể dừng giữa chừng, chỉnh sửa seed rồi chạy lại từ
đúng bước cần cập nhật mà không cần chạy lại toàn bộ notebook.

In [ ]:
# ============================================================
# THIẾT LẬP CHUNG: import thư viện và xác định đường dẫn thư mục
# ============================================================
from __future__ import annotations

from pathlib import Path
import re
import sys
import textwrap

import pandas as pd

# Notebook này nằm trong thư mục Seed_set_Prepare (News/Build_sentiment_label/
# Seed_set_Prepare), và toàn bộ dữ liệu seed nằm gọn trong chính thư mục này.
# Khi chạy notebook bình thường (Jupyter / VS Code), thư mục làm việc (cwd)
# mặc định chính là thư mục chứa file .ipynb, nên ta lấy PROJECT_ROOT bằng
# cách đi ngược lên 3 cấp: Seed_set_Prepare -> Build_sentiment_label -> News
# -> project3T (gốc repo).
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "Seed_set_Prepare":
    # Phòng trường hợp notebook được chạy từ cwd khác (vd chạy bằng nbconvert
    # từ thư mục gốc repo) thì thử suy ra đường dẫn trực tiếp từ gốc repo.
    fallback = NOTEBOOK_DIR / "News" / "Build_sentiment_label" / "Seed_set_Prepare"
    if fallback.exists():
        NOTEBOOK_DIR = fallback

PROJECT_ROOT = NOTEBOOK_DIR.parents[2]
if str(PROJECT_ROOT) not in sys.path:
    # Thêm gốc repo vào sys.path để import được module nội bộ
    # News.Build_sentiment_label.Common.matrix_csr_utils ở Bước 3.
    sys.path.insert(0, str(PROJECT_ROOT))

# Các thư mục / file dùng xuyên suốt notebook -- tất cả nằm trong
# Seed_set_Prepare/, không còn phụ thuộc thư mục nào ở ngoài.
DATA_DIR = NOTEBOOK_DIR / "Master_Dictionary"          # dữ liệu Loughran-McDonald
CATEGORIES_DIR = DATA_DIR / "categories"                # output Bước 1, input Bước 2
MASTER_DICTIONARY_PATH = DATA_DIR / "Loughran-McDonald_MasterDictionary_1993-2025.csv"
REPORT_DIR = DATA_DIR / "corpus_check"                  # output Bước 3 + Bước 4

VI_SEEDS_DIR = NOTEBOOK_DIR / "MD_seeds"                # output Bước 2 (seed dịch từ Master Dictionary)
RESOURCES_DIR = NOTEBOOK_DIR / "manual_seed"            # seed thủ công (không dịch từ Master Dictionary)
FINAL_SEED_DIR = NOTEBOOK_DIR / "final_seed"            # output Bước 4 (seed set cơ bản cuối cùng)

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("PROJECT_ROOT :", PROJECT_ROOT)

## Bước 1 — Trích 7 danh mục gốc từ Loughran–McDonald Master Dictionary

File `Loughran-McDonald_MasterDictionary_1993-2025.csv` chứa **toàn bộ từ tiếng Anh**
kèm 7 cột "flag" tương ứng 7 danh mục sentiment tài chính: Negative, Positive,
Uncertainty, Litigious, Strong_Modal, Weak_Modal, Constraining.

Quy ước của Master Dictionary: giá trị ở cột flag là **năm từ đó được thêm vào danh
mục** (ví dụ `2011`), giá trị `0` nghĩa là **không** thuộc danh mục đó. Vì vậy điều
kiện lọc đúng phải là `cột > 0`, không phải kiểu boolean True/False.

Bước này chỉ lọc & lưu riêng từng danh mục ra CSV (vẫn là tiếng Anh), chưa dịch sang
tiếng Việt — việc dịch nằm ở Bước 2.

In [ ]:
# ------------------------------------------------------------
# Bước 1.1 — Hàm đọc Master Dictionary và hàm lọc theo danh mục
# ------------------------------------------------------------

# 7 cột flag cần lọc. Tên phải khớp chính xác với tên cột trong file CSV gốc.
CATEGORY_COLUMNS = [
    "Negative",
    "Positive",
    "Uncertainty",
    "Litigious",
    "Strong_Modal",
    "Weak_Modal",
    "Constraining",
]


# Đọc file Master Dictionary gốc.
# Lưu ý quan trọng: file gốc có một từ thật là "NULL" (viết tắt pháp lý của
# "null and void"). Nếu dùng danh sách NA mặc định của pandas, từ này sẽ bị
# hiểu nhầm thành giá trị rỗng (NaN) và bị mất khỏi dữ liệu. Do đó phải tắt
# na_values mặc định (keep_default_na=False) và chỉ coi chuỗi rỗng "" là NaN.
def load_master_dictionary(path: Path = MASTER_DICTIONARY_PATH) -> pd.DataFrame:
    df = pd.read_csv(path, keep_default_na=False, na_values=[""])
    missing_columns = set(CATEGORY_COLUMNS + ["Word"]).difference(df.columns)
    if missing_columns:
        raise ValueError(f"Master Dictionary thiếu cột: {sorted(missing_columns)}")
    return df


# Lọc các từ thuộc 1 danh mục (cột flag > 0), sắp xếp giảm dần theo Word Count
# (tần suất xuất hiện trong bộ ngữ liệu gốc của Loughran-McDonald) để các từ
# phổ biến/quan trọng hơn hiện lên đầu danh sách.
def extract_category_words(df: pd.DataFrame, category: str) -> pd.DataFrame:
    matched = df.loc[df[category] > 0, ["Word", "Word Count", "Doc Count", category]].copy()
    matched = matched.rename(columns={category: f"{category}_year_added"})
    matched = matched.sort_values("Word Count", ascending=False).reset_index(drop=True)
    return matched

In [ ]:
# ------------------------------------------------------------
# Bước 1.2 — Chạy trích xuất cho cả 7 danh mục và lưu ra CSV
# ------------------------------------------------------------
master_df = load_master_dictionary()
print("Đường dẫn Master Dictionary:", MASTER_DICTIONARY_PATH)
print("Tổng số từ trong Master Dictionary:", len(master_df))

CATEGORIES_DIR.mkdir(parents=True, exist_ok=True)

for category in CATEGORY_COLUMNS:
    category_df = extract_category_words(master_df, category)
    output_path = CATEGORIES_DIR / f"{category.lower()}_master_dictionary.csv"
    category_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"\n{category}: {len(category_df)} từ")
    print("Top 15 theo Word Count:")
    print(category_df.head(15)[["Word", "Word Count", "Doc Count"]].to_string(index=False))
    print("Đã lưu:", output_path)

## Bước 2 — Dịch thủ công & gộp seed sang tiếng Việt

Mỗi danh mục tiếng Anh ở Bước 1 được **dịch thủ công** thành các cụm từ tiếng Việt
mang nghĩa tài chính (không dùng dịch máy tự động, vì từ tài chính tiếng Anh thường có
nhiều biến thể — số ít/số nhiều, danh từ/động từ/tính từ — cần gộp lại thành 1 seed
tiếng Việt duy nhất để tránh trùng lặp nghĩa).

**Nguyên tắc chọn & dịch** (áp dụng cho 7 bảng map ở các cell bên dưới):

- Chọn thủ công từ **top khoảng 150 từ theo Doc Count** của mỗi danh mục (kết quả
  Bước 1).
- **Bỏ qua các từ nối mang tính pháp lý/văn bản hợp đồng** (HEREIN, THEREOF, WHEREAS,
  HEREBY...) vì dịch riêng lẻ không mang nghĩa sentiment tài chính rõ ràng.
- **Gộp các biến thể của cùng một gốc từ** (ví dụ REQUIRE / REQUIRES / REQUIRED /
  REQUIRING) vào chung 1 seed tiếng Việt (`yêu_cầu_bắt_buộc`) để tránh trùng lặp.

Sau khi có bản đồ (mapping), bước này còn **validate ngược** lại với CSV ở Bước 1: đảm
bảo mọi từ tiếng Anh xuất hiện trong bản đồ thực sự tồn tại trong danh mục Master
Dictionary tương ứng — mục đích là bắt lỗi gõ nhầm chính tả khi tra cứu thủ công.

7 cell code tiếp theo là 7 bảng map (mỗi cell 1 danh mục: `{seed tiếng Việt: [danh sách
từ gốc tiếng Anh]}`), sau đó là các hàm xử lý dùng chung và cell chạy toàn bộ Bước 2.

In [ ]:
# Danh mục NEGATIVE — từ mang nghĩa tiêu cực về hoạt động/kết quả kinh doanh
# (thua lỗ, vi phạm, phá sản, rủi ro pháp lý mang tính bất lợi...).
NEGATIVE_MAP: dict[str, list[str]] = {
    "thua_lỗ": ["LOSS", "LOSSES", "LOST", "LOSE"],
    "bất_lợi": ["ADVERSELY", "ADVERSE", "NEGATIVELY", "NEGATIVE", "UNFAVORABLE"],
    "gây_hiểu_lầm": ["MISLEADING"],
    "kiện_tụng": ["LITIGATION"],
    "gian_lận": ["FRAUD"],
    "bỏ_sót_thông_tin": ["OMIT", "OMITTED", "OMISSIONS"],
    "thiếu_sót": ["DEFICIENCIES", "DEFICIENCY", "INSUFFICIENT", "INADEQUATE", "INEFFECTIVE"],
    "điểm_yếu": ["WEAKNESSES", "WEAKNESS"],
    "suy_giảm_giá_trị": ["IMPAIRMENT", "IMPAIRED", "IMPAIR", "IMPAIRMENTS"],
    "thâm_hụt": ["DEFICIT"],
    "không_thể": ["UNABLE", "INABILITY"],
    "thất_bại": ["FAILURE", "FAIL", "FAILED", "FAILS", "FAILURES"],
    "trình_bày_lại": ["RESTATED", "RESTATEMENT"],
    "chấm_dứt_hợp_đồng": ["TERMINATION", "TERMINATED", "TERMINATE", "TERMINATES"],
    "suy_giảm": ["DECLINE", "DECLINES", "DECLINED", "DECLINING"],
    "biến_động_mạnh": ["VOLATILITY", "VOLATILE"],
    "vỡ_nợ": ["DEFAULTS", "DEFAULT"],
    "khiếu_nại": ["CLAIMS"],
    "khó_khăn": ["DIFFICULT", "DIFFICULTIES", "DIFFICULTY"],
    "bị_phạt": ["PENALTIES", "PENALTY", "FINES"],
    "chưa_thanh_toán": ["UNPAID", "DELINQUENT", "ARREARS"],
    "nghi_ngờ_đáng_kể": ["DOUBTFUL", "DOUBT"],
    "vi_phạm": ["BREACH", "BREACHES", "VIOLATION", "VIOLATIONS", "VIOLATE", "VIOLATED"],
    "trì_hoãn": ["DELAY", "DELAYS", "DELAYED"],
    "thiếu_hụt": ["LACK", "ABSENCE"],
    "phá_sản": ["BANKRUPTCY", "INSOLVENCY", "DISSOLUTION"],
    "lo_ngại": ["CONCERN", "CONCERNS", "CAUTIONARY", "CAUTIONED"],
    "ngừng_hoạt_động": ["CEASE", "CEASED", "DISCONTINUED", "DISCONTINUE", "SUSPENDED", "SUSPENSION", "SUSPEND"],
    "tái_cấu_trúc": ["RESTRUCTURING"],
    "báo_cáo_sai": ["MISSTATEMENT", "MISSTATEMENTS", "INACCURATE", "INCORRECT", "ERROR", "ERRORS"],
    "bị_cáo_buộc": ["ALLEGED", "ALLEGING", "ALLEGATIONS"],
    "tồi_tệ": ["BAD", "POOR", "SEVERE"],
    "bào_chữa": ["DEFEND", "DEFENDING", "DEFENDANT", "DEFENDANTS", "PLAINTIFF", "PLAINTIFFS"],
    "ngoài_dự_kiến": ["UNANTICIPATED", "UNEXPECTED", "UNFORESEEN"],
    "hủy_bỏ": ["CANCELLED", "CANCELED", "CANCELLATION", "CANCEL"],
    "bị_đe_dọa": ["THREATENED"],
    "gián_đoạn": ["DISRUPTIONS", "DISRUPTION", "INTERRUPTION", "INTERRUPTIONS"],
    "mất_khả_năng_thu_hồi": ["UNCOLLECTIBLE"],
    "xung_đột": ["CONFLICT", "CONFLICTS"],
    "tranh_chấp": ["DISPUTES", "DISPUTE"],
    "thách_thức": ["CHALLENGES", "CHALLENGE", "CHALLENGING"],
    "không_được_phép": ["UNAUTHORIZED", "INVALID"],
    "chịu_thiệt_hại": ["SUFFER", "SUFFERED", "INJURY"],
    "lỗi_thời": ["OBSOLETE"],
    "buộc_phải_từ_chức": ["RESIGNATION"],
    "tốn_kém": ["COSTLY"],
    "xuống_dốc": ["DOWNTURN", "SLOW"],
    "không_thành_công": ["UNSUCCESSFUL"],
    "sơ_suất": ["NEGLIGENCE"],
    "thảm_họa": ["DISASTERS", "HAZARDOUS"],
    "bị_bác_bỏ": ["DISMISSED", "DENIED"],
    "sai_phạm": ["MISCONDUCT"],
    "không_nhất_quán": ["INCONSISTENT"],
    "chưa_giải_quyết": ["UNRESOLVED"],
    "tịch_thu_tài_sản": ["FORFEITURE", "FORFEITED", "FORFEITURES"],
    "xiết_nợ": ["FORECLOSURE"],
    "xâm_phạm": ["INFRINGEMENT"],
    "suy_thoái": ["DETERIORATE", "DETERIORATION"],
    "vấn_đề_phát_sinh": ["PROBLEM", "PROBLEMS"],
}

In [ ]:
# Danh mục POSITIVE — từ mang nghĩa tích cực về hoạt động/kết quả kinh doanh
# (tăng trưởng, cải thiện, lợi thế cạnh tranh, được đánh giá tốt...).
POSITIVE_MAP: dict[str, list[str]] = {
    "lợi_nhuận_tăng": ["GAIN", "GAINS", "GAINED", "GAINING"],
    "có_năng_lực": ["ABLE"],
    "tốt_nhất": ["BEST"],
    "cải_thiện": ["IMPROVEMENTS", "IMPROVEMENT", "IMPROVE", "IMPROVED", "IMPROVING", "IMPROVES"],
    "cơ_hội": ["OPPORTUNITIES", "OPPORTUNITY"],
    "thành_công": ["SUCCESSFUL", "SUCCESS", "SUCCESSFULLY", "SUCCEED", "SUCCEEDING", "SUCCEEDED"],
    "thuận_lợi": ["FAVORABLE", "FAVORABLY", "FAVORED", "FAVORING"],
    "đạt_được": ["ACHIEVE", "ACHIEVED", "ACHIEVING", "ACHIEVEMENT", "ACHIEVEMENTS", "ACHIEVES", "ACCOMPLISH", "ACCOMPLISHED", "ACCOMPLISHING", "ATTAIN", "ATTAINED", "ATTAINING", "ATTAINS", "ATTAINMENT"],
    "tiến_bộ": ["ADVANCES", "ADVANCING", "ADVANCEMENT", "ADVANCEMENTS", "PROGRESS", "PROGRESSES", "PROGRESSED"],
    "đáp_ứng": ["SATISFY", "SATISFIES", "SATISFIED", "SATISFACTION", "SATISFACTORY", "SATISFYING", "SATISFACTORILY"],
    "khả_năng_sinh_lời": ["PROFITABILITY", "PROFITABLE", "PROFITABLY"],
    "tốt": ["GOOD"],
    "tích_cực": ["POSITIVE", "POSITIVELY"],
    "độc_quyền": ["EXCLUSIVE", "EXCLUSIVELY", "EXCLUSIVITY"],
    "tạo_điều_kiện": ["ENABLE", "ENABLES", "ENABLING", "ENABLED"],
    "mạnh": ["STRONG", "STRONGER", "STRONGEST"],
    "tốt_hơn": ["BETTER"],
    "nâng_cao": ["ENHANCE", "ENHANCED", "ENHANCING", "ENHANCEMENT", "ENHANCEMENTS", "ENHANCES"],
    "dẫn_đầu": ["LEADING", "LEADERSHIP"],
    "lợi_thế": ["ADVANTAGE", "ADVANTAGES", "ADVANTAGEOUS"],
    "hài_lòng": ["SATISFIED", "PLEASED", "PLEASURE", "ENJOY", "ENJOYED", "ENJOYMENT"],
    "đảm_bảo": ["ASSURE", "ASSURED", "ASSURING"],
    "cao_nhất": ["HIGHEST"],
    "đầy_đủ": ["ADEQUATELY"],
    "vượt_trội": ["SUPERIOR", "EXCEPTIONAL", "EXCELLENT", "EXCELLENCE", "PREMIER", "EXEMPLARY"],
    "hiệu_quả": ["EFFICIENCY", "EFFICIENT", "EFFICIENTLY", "EFFICIENCIES"],
    "sức_mạnh": ["STRENGTH", "STRENGTHS", "STRENGTHEN", "STRENGTHENING", "STRENGTHENED", "STRENGTHENS"],
    "giá_trị": ["VALUABLE"],
    "mong_muốn": ["DESIRED", "DESIRABLE"],
    "hấp_dẫn": ["ATTRACTIVE", "ATTRACTIVENESS"],
    "ổn_định": ["STABLE", "STABILITY", "STABILIZE", "STABILIZATION", "STABILIZED", "STABILIZING"],
    "đổi_mới": ["INNOVATIVE", "INNOVATION", "INNOVATIONS", "INNOVATE"],
    "liêm_chính": ["INTEGRITY"],
    "minh_bạch": ["TRANSPARENCY"],
    "sáng_chế": ["INVENTIONS", "INVENTION", "INVENTOR"],
    "hợp_tác": ["ALLIANCES", "ALLIANCE", "COLLABORATION", "COLLABORATIVE", "COLLABORATIONS", "COLLABORATE", "COLLABORATING", "COLLABORATOR", "COLLABORATORS"],
    "được_hưởng_lợi": ["BENEFITED", "BENEFITING"],
    "dễ_dàng": ["EASILY", "EASY", "EASIER"],
    "vinh_dự": ["HONOR", "HONORED"],
    "từ_thiện": ["CHARITABLE"],
    "phần_thưởng": ["REWARD", "REWARDING"],
    "hoàn_hảo": ["PERFECT", "PERFECTED"],
    "được_ưa_chuộng": ["POPULAR", "POPULARITY"],
    "được_trao_quyền": ["EMPOWERED", "EMPOWER"],
    "mang_tính_xây_dựng": ["CONSTRUCTIVE"],
    "thân_thiện": ["FRIENDLY"],
    "sáng_tạo": ["CREATIVE"],
    "chủ_động": ["PROACTIVE", "PROACTIVELY", "DILIGENT", "DILIGENTLY"],
    "tự_tin": ["CONFIDENT"],
    "lạc_quan": ["OPTIMISTIC"],
    "chiến_thắng": ["WIN", "WINNING"],
    "lấy_lại": ["REGAIN"],
    "khác_biệt_nổi_bật": ["DISTINCTIVE", "DISTINCTION"],
    "đột_phá": ["BREAKTHROUGH"],
    "lý_tưởng": ["IDEAL"],
    "suôn_sẻ": ["SMOOTH"],
    "xứng_đáng": ["MERITORIOUS"],
}

In [ ]:
# Danh mục UNCERTAINTY — từ thể hiện sự không chắc chắn, ước lượng, mơ hồ
# (chưa xác định, dự kiến, biến động, phụ thuộc điều kiện...).
UNCERTAINTY_MAP: dict[str, list[str]] = {
    "xấp_xỉ": ["APPROXIMATELY", "APPROXIMATE", "APPROXIMATES", "APPROXIMATED", "APPROXIMATING", "ROUGHLY"],
    "rủi_ro": ["RISK", "RISKS"],
    "rủi_ro_cao": ["RISKY"],
    "cho_rằng": ["BELIEVE", "BELIEVES", "BELIEVED"],
    "giả_định": ["ASSUMPTIONS", "ASSUMPTION", "ASSUME", "ASSUMED", "ASSUMING", "ASSUMES"],
    "không_chắc_chắn": ["UNCERTAINTIES", "UNCERTAINTY", "UNCERTAIN"],
    "dự_kiến": ["ANTICIPATED", "ANTICIPATE", "ANTICIPATES", "ANTICIPATION", "ANTICIPATING"],
    "tình_huống_bất_ngờ": ["CONTINGENCIES", "CONTINGENCY", "CONTINGENT", "CONTINGENTLY"],
    "đang_chờ_xử_lý": ["PENDING", "UNSETTLED"],
    "phụ_thuộc": ["DEPENDENT", "DEPENDENCE", "DEPENDENCY"],
    "biến_động": ["FLUCTUATIONS", "FLUCTUATE", "FLUCTUATION", "FLUCTUATING", "FLUCTUATED", "FLUCTUATES"],
    "biến_động_mạnh": ["VOLATILITY", "VOLATILE", "VOLATILITIES"],
    "thay_đổi_thất_thường": ["VARIABLE", "VARY", "VARYING", "VARIES", "VARIED", "VARIATION", "VARIATIONS", "VARIABILITY", "VARIANCE", "VARIANCES", "VARIANTS"],
    "điều_chỉnh_lại": ["REVISED", "REVISE"],
    "dự_báo": ["PREDICT", "PREDICTED", "PREDICTS", "PREDICTING", "PREDICTION", "PREDICTIONS", "PREDICTIVE"],
    "khó_dự_đoán": ["UNPREDICTABLE", "UNPREDICTABILITY"],
    "còn_nghi_vấn": ["DOUBTFUL", "DOUBT", "DOUBTS"],
    "khả_năng_xảy_ra": ["LIKELIHOOD"],
    "chưa_xác_định": ["UNKNOWN", "UNIDENTIFIED", "UNSPECIFIED", "UNDESIGNATED"],
    "tài_sản_vô_hình": ["INTANGIBLE", "INTANGIBLES"],
    "sơ_bộ": ["PRELIMINARY", "PRELIMINARILY"],
    "không_xác_định_thời_hạn": ["INDEFINITE", "INDEFINITELY"],
    "bất_ngờ": ["UNEXPECTED", "UNEXPECTEDLY", "SUDDEN"],
    "bất_thường": ["UNUSUAL", "UNUSUALLY"],
    "có_điều_kiện": ["CONDITIONAL", "CONDITIONALLY"],
    "bất_ổn": ["INSTABILITY"],
    "cần_làm_rõ": ["CLARIFICATION", "UNCLEAR"],
    "biện_pháp_phòng_ngừa": ["PRECAUTIONS"],
    "đánh_giá_lại": ["REASSESS", "REASSESSED", "REASSESSMENT", "RECONSIDER"],
    "ngoài_kế_hoạch": ["UNPLANNED"],
    "chưa_được_chứng_minh": ["UNPROVEN", "UNPROVED"],
    "không_thể_xác_định": ["INDETERMINATE"],
    "mơ_hồ": ["AMBIGUITY", "AMBIGUITIES"],
    "được_cho_là": ["PRESUMED", "PRESUMPTION"],
    "ít_có_khả_năng": ["IMPROBABLE"],
    "không_thể_đánh_giá_được": ["NONASSESSABLE"],
    "thay_đổi_điều_khoản": ["ALTERATION", "ALTERATIONS"],
}

In [ ]:
# Danh mục LITIGIOUS — từ liên quan pháp lý, tố tụng, kiện tụng, hợp đồng
# (tòa án, khiếu nại, quy định, luật sư, trọng tài...).
LITIGIOUS_MAP: dict[str, list[str]] = {
    "pháp_lý": ["LEGAL", "LEGALLY"],
    "hợp_pháp": ["LAWFUL", "LAWFULLY"],
    "bất_hợp_pháp": ["UNLAWFUL"],
    "sửa_đổi": ["AMENDED", "AMENDMENT", "AMENDMENTS", "AMEND", "AMENDS", "AMENDING"],
    "quy_định": ["REGULATION", "REGULATIONS"],
    "cơ_quan_quản_lý": ["REGULATORY", "REGULATED", "REGULATE", "REGULATING", "REGULATES", "REGULATORS"],
    "luật_pháp": ["LAWS", "LAW"],
    "lập_pháp": ["LEGISLATION", "LEGISLATIVE"],
    "kiện_tụng": ["LITIGATION"],
    "vụ_kiện": ["LAWSUIT", "LAWSUITS"],
    "hợp_đồng": ["CONTRACTS", "CONTRACT", "CONTRACTUAL", "CONTRACTUALLY", "CONTRACTED", "CONTRACTING"],
    "dàn_xếp_giải_quyết": ["SETTLEMENT", "SETTLEMENTS"],
    "tòa_án": ["COURT", "COURTS"],
    "tư_pháp": ["JUDICIAL", "JUSTICE"],
    "khiếu_nại": ["CLAIMS", "CLAIM"],
    "chấp_thuận": ["CONSENT", "CONSENTS", "CONSENTED"],
    "luật_sư": ["COUNSEL", "ATTORNEY", "ATTORNEYS"],
    "vi_phạm_hợp_đồng": ["BREACH", "BREACHES", "BREACHED"],
    "ban_hành": ["PROMULGATED"],
    "bồi_thường_thiệt_hại": ["INDEMNIFICATION", "INDEMNIFY", "INDEMNIFIED", "INDEMNITY", "INDEMNITIES"],
    "trợ_cấp_thôi_việc": ["SEVERANCE"],
    "cáo_buộc": ["ALLEGED", "ALLEGING", "ALLEGATIONS", "ALLEGES", "ALLEGEDLY", "ALLEGE"],
    "làm_chứng": ["WITNESS"],
    "kháng_cáo": ["APPEAL", "APPEALS", "APPEALED"],
    "được_diễn_giải_theo": ["CONSTRUED"],
    "có_hiệu_lực_thi_hành": ["ENFORCEABLE", "ENFORCEABILITY"],
    "không_thể_thi_hành": ["UNENFORCEABLE", "UNENFORCEABILITY"],
    "hình_sự": ["CRIMINAL"],
    "bị_đơn": ["DEFENDANTS", "DEFENDANT"],
    "trọng_tài": ["ARBITRATION", "ARBITRATOR"],
    "nguyên_đơn": ["PLAINTIFFS", "PLAINTIFF"],
    "đơn_kiện": ["PETITION"],
    "không_thể_hủy_ngang": ["IRREVOCABLE", "IRREVOCABLY"],
    "bảo_lãnh": ["SURETY"],
    "lệnh_cấm": ["INJUNCTIVE", "INJUNCTION", "INJUNCTIONS"],
    "cố_ý_vi_phạm": ["WILLFUL"],
    "phán_quyết": ["RULING", "RULINGS"],
    "gây_bất_lợi": ["PREJUDICE"],
    "sắc_lệnh": ["DECREE"],
    "thu_hồi": ["REVOCATION"],
    "bồi_thẩm_đoàn": ["JURY"],
    "hành_vi_gây_thiệt_hại": ["TORT"],
    "hủy_bỏ_hợp_đồng": ["RESCISSION"],
    "thừa_nhận": ["ADMISSION"],
}

In [ ]:
# Danh mục STRONG_MODAL — từ thể hiện mức độ chắc chắn/khẳng định cao
# (chắc chắn, luôn luôn, cao nhất, không thể tranh cãi...).
STRONG_MODAL_MAP: dict[str, list[str]] = {
    "chắc_chắn_sẽ": ["WILL"],
    "bắt_buộc_phải": ["MUST"],
    "tốt_nhất": ["BEST"],
    "cao_nhất": ["HIGHEST"],
    "thấp_nhất": ["LOWEST"],
    "không_bao_giờ": ["NEVER"],
    "luôn_luôn": ["ALWAYS"],
    "rõ_ràng": ["CLEARLY"],
    "mạnh_mẽ": ["STRONGLY"],
    "không_thể_tranh_cãi": ["UNDISPUTED"],
    "dứt_khoát": ["DEFINITIVELY"],
    "vô_song": ["UNPARALLELED"],
    "chắc_chắn": ["DEFINITELY"],
    "không_nghi_ngờ_gì": ["UNDOUBTEDLY"],
    "minh_bạch_tuyệt_đối": ["UNEQUIVOCALLY", "UNEQUIVOCAL"],
}

In [ ]:
# Danh mục WEAK_MODAL — từ thể hiện mức độ chắc chắn thấp, dè dặt
# (có thể, có lẽ, dường như, đôi khi...).
WEAK_MODAL_MAP: dict[str, list[str]] = {
    "có_thể": ["MAY", "POSSIBLY", "MAYBE"],
    "có_thể_sẽ": ["COULD"],
    "có_lẽ_sẽ": ["MIGHT"],
    "tùy_thuộc_vào": ["DEPENDING", "DEPENDED"],
    "phụ_thuộc_vào": ["DEPEND", "DEPENDS"],
    "không_chắc_chắn": ["UNCERTAIN", "UNCERTAINLY"],
    "dường_như": ["APPEARS", "APPEARED"],
    "có_vẻ_như": ["APPEARING"],
    "gần_như": ["NEARLY", "ALMOST"],
    "đôi_khi": ["SOMETIMES"],
    "thỉnh_thoảng": ["OCCASIONALLY"],
    "phần_nào": ["SOMEWHAT"],
    "cho_thấy_khả_năng": ["SUGGEST"],
    "gợi_ý_rằng": ["SUGGESTS"],
    "có_lẽ": ["PERHAPS"],
    "hình_như": ["APPARENTLY"],
    "hiếm_khi": ["SELDOM", "SELDOMLY"],
}

In [ ]:
# Danh mục CONSTRAINING — từ thể hiện ràng buộc, nghĩa vụ, hạn chế
# (yêu cầu bắt buộc, cam kết, bị hạn chế, cấm, ràng buộc hợp đồng...).
CONSTRAINING_MAP: dict[str, list[str]] = {
    "yêu_cầu_bắt_buộc": ["REQUIREMENTS", "REQUIREMENT", "REQUIRED", "REQUIRES", "REQUIRE", "REQUIRING"],
    "nghĩa_vụ": ["OBLIGATIONS", "OBLIGATION", "OBLIGATED", "OBLIGATE", "OBLIGATES", "OBLIGATING", "OBLIGATORY", "OBLIGED"],
    "cam_kết": ["COMMITMENTS", "COMMITMENT", "COMMITTED", "COMMIT", "COMMITS", "COMMITTING"],
    "bị_hạn_chế": ["RESTRICTED", "RESTRICTIONS", "RESTRICT", "RESTRICTS", "RESTRICTING", "RESTRICTION", "RESTRICTIVE"],
    "được_phép": ["PERMITTED", "PERMITTING", "PERMISSIBLE", "PERMISSION", "PERMISSIONS"],
    "giới_hạn": ["LIMIT", "LIMITS", "LIMITING"],
    "tuân_thủ": ["COMPLY", "ABIDE"],
    "ngăn_cản": ["PREVENT", "PREVENTED", "PREVENTING", "PREVENTS"],
    "áp_đặt": ["IMPOSED", "IMPOSE", "IMPOSES", "IMPOSING", "IMPOSITION", "IMPOSITIONS"],
    "cầm_cố": ["PLEDGED", "PLEDGE", "PLEDGES", "PLEDGING"],
    "bắt_buộc": ["MANDATORY", "MANDATED", "MANDATE", "MANDATES", "MANDATING", "COMPULSORY", "COMPULSION"],
    "cấm": ["PROHIBITED", "PROHIBIT", "PROHIBITS", "PROHIBITION", "PROHIBITING", "PROHIBITIONS", "PROHIBITIVE", "PROHIBITIVELY", "FORBIDDEN"],
    "ràng_buộc": ["CONSTRAINTS", "CONSTRAINED", "CONSTRAIN", "CONSTRAINT"],
    "bị_ràng_buộc": ["BOUND"],
    "không_thể_hủy_ngang": ["IRREVOCABLE", "IRREVOCABLY"],
    "loại_trừ_khả_năng": ["PRECLUDE", "PRECLUDED", "PRECLUDES", "PRECLUDING"],
    "nghiêm_ngặt": ["STRICT", "STRICTLY", "STRICTER", "STRICTEST"],
    "không_sẵn_có": ["UNAVAILABLE", "UNAVAILABILITY"],
    "kìm_hãm": ["INHIBIT", "INHIBITING", "INHIBITED", "INHIBITS"],
    "không_thể_hủy_bỏ": ["NONCANCELABLE", "NONCANCELLABLE"],
    "ký_quỹ": ["ESCROW", "ESCROWED", "ESCROWS"],
    "buộc_phải": ["COMPELLING", "COMPEL", "COMPELLED"],
    "đòi_hỏi_phải": ["NECESSITATE", "NECESSITATED", "NECESSITATING", "NECESSITATES"],
    "giới_hạn_trong_phạm_vi": ["CONFINED", "CONFINES"],
    "mắc_nợ": ["INDEBTED"],
    "điều_kiện_tiên_quyết": ["PRECONDITION"],
}

In [ ]:
# Gộp cả 7 bảng map lại thành 1 dict tổng, key là tên danh mục (chữ thường,
# khớp với hậu tố tên file .txt / .csv sẽ ghi ra ở các bước sau).
CATEGORY_SEEDS: dict[str, dict[str, list[str]]] = {
    "negative": NEGATIVE_MAP,
    "positive": POSITIVE_MAP,
    "uncertainty": UNCERTAINTY_MAP,
    "litigious": LITIGIOUS_MAP,
    "strong_modal": STRONG_MODAL_MAP,
    "weak_modal": WEAK_MODAL_MAP,
    "constraining": CONSTRAINING_MAP,
}

### Các hàm xử lý của Bước 2

- `validate_against_master_dictionary`: kiểm tra mọi từ tiếng Anh trong 1 bảng map có
  thực sự nằm trong CSV danh mục tương ứng (kết quả Bước 1) hay không. Nếu có từ "lạ"
  (gõ nhầm chính tả, hoặc nhầm danh mục) thì báo lỗi ngay để sửa, tránh việc seed sai bị
  lọt xuống các bước sau mà không ai phát hiện.
- `write_seed_txt`: ghi danh sách seed tiếng Việt (chỉ lấy key của map, bỏ phần từ tiếng
  Anh) ra file `.txt`, các seed cách nhau bởi dấu phẩy và tự động xuống dòng cho dễ đọc
  (không ảnh hưởng tới cách đọc lại file ở Bước 3, vì Bước 3 tách theo cả dấu phẩy lẫn
  xuống dòng).
- `write_mapping_csv`: ghi lại toàn bộ bảng map (seed tiếng Việt kèm danh sách từ tiếng
  Anh gốc) ra CSV để sau này tra cứu nguồn gốc của từng seed khi cần.

In [ ]:
# ------------------------------------------------------------
# Bước 2.1 — Hàm validate và hàm ghi output (seed .txt + mapping .csv)
# ------------------------------------------------------------


# Kiểm tra mọi từ tiếng Anh trong seed_map có thực sự tồn tại trong danh mục
# tương ứng của Master Dictionary hay không (tránh gõ nhầm chính tả khi tra
# cứu thủ công từ danh sách in ra màn hình ở Bước 1).
def validate_against_master_dictionary(category: str, seed_map: dict[str, list[str]]) -> None:
    category_csv = CATEGORIES_DIR / f"{category}_master_dictionary.csv"
    known_words = set(pd.read_csv(category_csv)["Word"])
    all_mapped_words = {word for words in seed_map.values() for word in words}
    unknown_words = sorted(all_mapped_words.difference(known_words))
    if unknown_words:
        raise ValueError(
            f"[{category}] {len(unknown_words)} từ không tồn tại trong danh mục "
            f"Master Dictionary '{category}': {unknown_words}"
        )


# Ghi danh sách seed tiếng Việt ra .txt, các seed cách nhau bởi dấu phẩy, tự
# động xuống dòng ở độ rộng `width` ký tự cho dễ đọc bằng mắt thường.
def write_seed_txt(vi_terms: list[str], output_path: Path, width: int = 79) -> None:
    text = ", ".join(vi_terms)
    wrapped_lines = textwrap.wrap(
        text, width=width, break_long_words=False, break_on_hyphens=False
    )
    output_path.write_text("\n".join(wrapped_lines) + "\n", encoding="utf-8")


# Ghi toàn bộ bảng map (seed tiếng Việt + các từ tiếng Anh gốc) ra CSV để tra
# cứu nguồn gốc của seed khi cần.
def write_mapping_csv(seed_map: dict[str, list[str]], output_path: Path) -> None:
    rows = [
        {"vietnamese_term": vi_term, "english_words": "; ".join(en_words)}
        for vi_term, en_words in seed_map.items()
    ]
    pd.DataFrame(rows).to_csv(output_path, index=False, encoding="utf-8-sig")

In [ ]:
# ------------------------------------------------------------
# Bước 2.2 — Chạy validate + ghi output cho cả 7 danh mục
# ------------------------------------------------------------
VI_SEEDS_DIR.mkdir(parents=True, exist_ok=True)

for category, seed_map in CATEGORY_SEEDS.items():
    validate_against_master_dictionary(category, seed_map)

    vi_terms = list(seed_map.keys())
    seed_txt_path = VI_SEEDS_DIR / f"{category}_word.txt"
    write_seed_txt(vi_terms, seed_txt_path)

    mapping_csv_path = CATEGORIES_DIR / f"{category}_vi_mapping.csv"
    write_mapping_csv(seed_map, mapping_csv_path)

    total_english_words = sum(len(words) for words in seed_map.values())
    print(
        f"{category}: {len(vi_terms)} seed tiếng Việt, gộp từ "
        f"{total_english_words} từ tiếng Anh trong Master Dictionary"
    )
    print("  ->", seed_txt_path)
    print("  ->", mapping_csv_path)

## Bước 3 — Đối chiếu seed tiếng Việt với corpus tin tức thật

Bước này build bảng tần suất n-gram (1 đến 3 từ) từ **toàn bộ corpus tin tức đã
tokenize**, rồi đối chiếu từng seed tiếng Việt ở Bước 2 xem có thực sự xuất hiện trong
corpus hay không, kèm tần suất xuất hiện:

- `tf` (term frequency): tổng số lần seed xuất hiện trong toàn corpus.
- `df` (document frequency): số bài báo có chứa seed đó (ít nhất 1 lần).

Phạm vi n-gram (1-3) được chọn **khớp với bước resolve seed** dự kiến dùng trong pipeline
xây dictionary theo PMI ở bước tiếp theo (`SEED_MIN_N=1`, `SEED_MAX_N=3`), để đảm bảo
nhất quán trong toàn bộ pipeline — seed nào không tìm thấy ở đây thì cũng sẽ không thể
dùng để tính PMI ở bước sau.

Kết quả cuối cùng được lưu ra `data/corpus_check/vi_seeds_corpus_presence.csv`, dùng để
rà soát và chỉnh lại seed nào bị viết sai/không khớp cách tokenizer tách từ trước khi
đưa seed vào bước xây dictionary PMI.

In [ ]:
# ------------------------------------------------------------
# Bước 3.1 — Import tiện ích build n-gram từ corpus + các hàm phụ trợ
# ------------------------------------------------------------
from News.Build_sentiment_label.Common.matrix_csr_utils import (
    DEFAULT_TOKENIZED_NEWS_PATH,
    NGRAM_SEPARATOR,
    build_ngram_terms_with_summary,
)

# Cùng phạm vi n-gram với bước resolve seed trong build_sentiment_dictionary_pmi.py
# (SEED_MIN_N=1, SEED_MAX_N=3) để đảm bảo nhất quán trong toàn pipeline.
CHECK_MIN_N = 1
CHECK_MAX_N = 3

SEED_CATEGORIES = [
    "negative",
    "positive",
    "uncertainty",
    "litigious",
    "strong_modal",
    "weak_modal",
    "constraining",
]


# Đọc file seed .txt và tách thành danh sách từ/cụm từ riêng lẻ. Tách theo cả
# dấu phẩy lẫn ký tự xuống dòng vì write_seed_txt ở Bước 2 dùng cả hai để định
# dạng file cho dễ đọc.
def load_seed_words(path: Path) -> list[str]:
    text = path.read_text(encoding="utf-8")
    items = [item.strip() for item in re.split(r"[,\n]", text)]
    return [item for item in items if item]


# Chuẩn hóa 1 n-gram từ bảng tần suất corpus (các từ cách nhau bởi dấu cách)
# về cùng định dạng với seed tiếng Việt (các từ cách nhau bởi dấu gạch dưới)
# để có thể so khớp trực tiếp.
def normalize_for_seed_matching(term: str, separator: str = NGRAM_SEPARATOR) -> str:
    return term.replace(separator, "_")

In [ ]:
# ------------------------------------------------------------
# Bước 3.2 — Build bảng tần suất n-gram của corpus và đối chiếu với seed
# ------------------------------------------------------------
print("Đang xây dựng bảng tần suất n-gram (1-3) từ toàn bộ corpus...")
print("Corpus:", DEFAULT_TOKENIZED_NEWS_PATH)
corpus_ngram_terms_df, summary = build_ngram_terms_with_summary(
    path=DEFAULT_TOKENIZED_NEWS_PATH,
    min_n=CHECK_MIN_N,
    max_n=CHECK_MAX_N,
)
print("Tổng số văn bản:", summary["total_documents"])
print("Số n-gram duy nhất (1-3 gram) trong corpus:", summary["unique_ngrams"])

# Chuẩn hóa key để so khớp, rồi gộp (sum) df/tf theo từng n-gram đã chuẩn hóa
# — phòng trường hợp cùng 1 seed match nhiều biến thể phân tách khác nhau.
corpus_ngram_terms_df = corpus_ngram_terms_df.copy()
corpus_ngram_terms_df["seed_key"] = corpus_ngram_terms_df["term"].apply(
    normalize_for_seed_matching
)
lookup_df = corpus_ngram_terms_df.groupby("seed_key").agg(
    df=("df", "sum"), tf=("tf", "sum")
)
lookup = lookup_df.to_dict(orient="index")

REPORT_DIR.mkdir(parents=True, exist_ok=True)
all_rows = []
print()
for category in SEED_CATEGORIES:
    seed_path = VI_SEEDS_DIR / f"{category}_word.txt"
    seed_words = load_seed_words(seed_path)

    found = 0
    missing_terms = []
    for seed in seed_words:
        key = normalize_for_seed_matching(seed)
        hit = lookup.get(key)
        if hit is not None:
            found += 1
            all_rows.append(
                {
                    "category": category,
                    "term": seed,
                    "found_in_corpus": True,
                    "df": int(hit["df"]),
                    "tf": int(hit["tf"]),
                }
            )
        else:
            missing_terms.append(seed)
            all_rows.append(
                {
                    "category": category,
                    "term": seed,
                    "found_in_corpus": False,
                    "df": 0,
                    "tf": 0,
                }
            )

    total = len(seed_words)
    print(f"{category}: {found}/{total} term xuất hiện trong corpus (>=1 lần)")
    if missing_terms:
        print(f"  KHÔNG xuất hiện ({len(missing_terms)}):", ", ".join(missing_terms))

report_df = pd.DataFrame(all_rows).sort_values(
    by=["category", "found_in_corpus", "df"], ascending=[True, True, False]
)
report_path = REPORT_DIR / "vi_seeds_corpus_presence.csv"
report_df.to_csv(report_path, index=False, encoding="utf-8-sig")
print("\nĐã lưu báo cáo đầy đủ:", report_path)
report_df

## Bước 4 — Gộp 2 bộ seed (thủ công + Master Dictionary) thành 1 seed set cơ bản cuối cùng

Hiện có 2 nguồn seed độc lập cho cùng 7 danh mục:

- **`manual_seed/`** — seed tiếng Việt tự xây dựng thủ công, không dịch từ Master Dictionary.
- **`MD_seeds/`** (kết quả Bước 2) — seed dịch từ Loughran-McDonald Master Dictionary.

Trước khi chạy PMI mở rộng lần đầu, cần **gộp 2 bộ này thành 1 seed set cơ bản duy nhất** (union, loại trùng lặp), rồi **đối chiếu lại với corpus thật** — vì seed trong `manual_seed/` chưa từng được kiểm tra ở Bước 3 (khi đó chỉ kiểm tra seed từ Master Dictionary).

Nguyên tắc lọc: **seed nào có `df = 0`** (không xuất hiện lần nào trong toàn bộ corpus tin tức) **sẽ bị loại khỏi seed set dùng để chạy PMI** — vì PMI không tính được gì có ý nghĩa từ 1 từ chưa từng đồng xuất hiện với bất kỳ candidate term nào. Seed bị loại vẫn được ghi lại trong báo cáo kèm lý do, không bị xóa âm thầm.

In [ ]:
# ------------------------------------------------------------
# Bước 4.1 — Hàm gộp 2 nguồn seed (Resources + Master Dictionary) cho 1
# danh mục, loại trùng lặp nhưng vẫn giữ lại nguồn gốc của từng seed.
# ------------------------------------------------------------


def merge_two_seed_sources(category: str) -> tuple[list[str], set[str], set[str]]:
    resources_words = load_seed_words(RESOURCES_DIR / f"{category}_word.txt")
    md_words = load_seed_words(VI_SEEDS_DIR / f"{category}_word.txt")

    resources_set = set(resources_words)
    md_set = set(md_words)

    # Ưu tiên thứ tự: seed Master Dictionary trước (đã qua validate ở Bước 2
    # và check corpus ở Bước 3), sau đó thêm seed Resources chưa trùng.
    merged_order: list[str] = []
    seen: set[str] = set()
    for term in md_words + resources_words:
        if term not in seen:
            seen.add(term)
            merged_order.append(term)

    return merged_order, resources_set, md_set

In [ ]:
# ------------------------------------------------------------
# Bước 4.2 — Với mỗi danh mục: gộp 2 nguồn, tra df/tf từ `lookup` (đã build
# ở Bước 3 — cần chạy Bước 3 trước khi chạy cell này), loại seed có df = 0,
# ghi seed set cuối cùng ra vi_seeds_final/ + báo cáo gộp chi tiết.
# ------------------------------------------------------------
FINAL_SEED_DIR.mkdir(parents=True, exist_ok=True)

all_merge_rows = []
print(f"{'category':13s} {'resources':>9s} {'master_dict':>11s} {'union':>6s} {'loại(df=0)':>11s} {'final':>6s}")
for category in SEED_CATEGORIES:
    merged_order, resources_set, md_set = merge_two_seed_sources(category)

    kept_terms = []
    removed_terms = []
    for term in merged_order:
        key = normalize_for_seed_matching(term)
        hit = lookup.get(key)
        df_val = int(hit["df"]) if hit is not None else 0
        tf_val = int(hit["tf"]) if hit is not None else 0

        if term in resources_set and term in md_set:
            source = "both"
        elif term in resources_set:
            source = "resources"
        else:
            source = "master_dict"

        kept = df_val > 0
        all_merge_rows.append(
            {"category": category, "term": term, "source": source, "df": df_val, "tf": tf_val, "kept": kept}
        )
        (kept_terms if kept else removed_terms).append(term)

    write_seed_txt(kept_terms, FINAL_SEED_DIR / f"{category}_word.txt")

    print(
        f"{category:13s} {len(resources_set):9d} {len(md_set):11d} "
        f"{len(merged_order):6d} {len(removed_terms):11d} {len(kept_terms):6d}"
    )
    if removed_terms:
        print(f"  -> loại (df=0): {', '.join(removed_terms)}")

merge_report_df = pd.DataFrame(all_merge_rows).sort_values(
    by=["category", "kept", "df"], ascending=[True, True, False]
)
merge_report_path = REPORT_DIR / "vi_seeds_merged_report.csv"
merge_report_df.to_csv(merge_report_path, index=False, encoding="utf-8-sig")
print("\nĐã lưu báo cáo gộp seed:", merge_report_path)
print("Đã lưu seed set cuối cùng vào thư mục:", FINAL_SEED_DIR)
merge_report_df

## Kết quả & bước tiếp theo

Sau khi chạy xong notebook, `Seed_set_Prepare/` sẽ có đầy đủ:

- `Master_Dictionary/categories/*_master_dictionary.csv` — 7 danh mục gốc tiếng Anh (Bước 1).
- `MD_seeds/*_word.txt` — seed tiếng Việt dịch từ Master Dictionary (Bước 2).
- `Master_Dictionary/categories/*_vi_mapping.csv` — bản đồ seed <-> từ gốc tiếng Anh (Bước 2).
- `Master_Dictionary/corpus_check/vi_seeds_corpus_presence.csv` — đối chiếu seed Master Dictionary với corpus (Bước 3).
- `final_seed/*_word.txt` — **seed set cơ bản cuối cùng** (gộp `manual_seed/` + `MD_seeds/`, đã loại seed `df=0`) (Bước 4).
- `Master_Dictionary/corpus_check/vi_seeds_merged_report.csv` — nguồn gốc + df/tf từng seed sau khi gộp (Bước 4).

**Việc cần làm tiếp theo (Bước 5, chưa có trong notebook này)**: dùng `final_seed/*_word.txt` làm seed anchor để chạy PMI với `candidate_ngram_terms`, mở rộng seed set cơ bản thành 1 bộ từ điển sentiment tiếng Việt đầy đủ hơn — có áp dụng ngưỡng lọc (df/tf tối thiểu cho candidate term, tính PMI ở cấp câu, margin PMI giữa 2 category cao nhất) để giảm nhiễu.